In [ ]:

!pip install open_clip_torch -q

import torch, os, glob, random, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import open_clip
from torchvision import transforms
from PIL import Image
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device.upper()}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:


PROMPTS = [
    "chest X-ray showing healthy lungs, grayscale, medical imaging",
    "chest X-ray showing pneumonia infection, grayscale, medical imaging",
    "chest X-ray showing lung nodule, grayscale, medical imaging"
]
SEEDS  = [42, 7, 123]
STEPS  = 5000       
LR     = 0.02       
NOISE  = 0.05       
SIGMA  = 0.5

def score_fn(x, prompt):
    xi = x.detach().clone().requires_grad_(True)
    r  = torch.nn.functional.interpolate(xi, (224,224), mode="bilinear", align_corners=False)
    n  = (r.clamp(0,1) - CLIP_MEAN) / CLIP_STD
    tok = tokenizer([prompt]).to(device)
    i_f = model.encode_image(n)
    t_f = model.encode_text(tok)
    i_f = i_f / i_f.norm(dim=-1, keepdim=True)
    t_f = t_f / t_f.norm(dim=-1, keepdim=True)
    clip_s = (i_f * t_f).sum() * 6.0   # was 4.0 — stronger guidance
    phi = model.encode_image(n)
    phi = phi / phi.norm(dim=-1, keepdim=True)
    k_v = torch.exp(-(((phi - feats)**2).sum(-1)) / (2*SIGMA**2))
    kern_s = (alpha.squeeze() * k_v).sum() * 0.8   # was 0.4 — stronger domain pull
    (clip_s + kern_s).backward()
    return xi.grad.detach()

results = []

for idx, (prompt, seed) in enumerate(zip(PROMPTS, SEEDS)):
    print("Generating image " + str(idx+1) + "/3: " + prompt[:50])
    torch.manual_seed(seed)
    x    = torch.rand(1, 3, IMG_SIZE, IMG_SIZE).to(device) * 0.3 + 0.35  # start closer to mid-gray
    sims = []
    tok  = tokenizer([prompt]).to(device)

    for t in range(STEPS + 1):
        noise_t = NOISE * (1 - t / STEPS) * 0.5   # noise decays faster = cleaner end
        s = score_fn(x, prompt)
        # normalize score to prevent exploding
        s = s / (s.abs().mean() + 1e-8) * 0.1
        x = (x + LR * s + torch.randn_like(x) * noise_t).clamp(0, 1)

        if t % 100 == 0:
            with torch.no_grad():
                r   = torch.nn.functional.interpolate(x, (224,224), mode="bilinear", align_corners=False)
                nn_ = (r - CLIP_MEAN) / CLIP_STD
                i_f = model.encode_image(nn_)
                t_f = model.encode_text(tok)
                i_f = i_f / i_f.norm(dim=-1, keepdim=True)
                t_f = t_f / t_f.norm(dim=-1, keepdim=True)
                sim = (i_f * t_f).sum().item()
            sims.append((t, sim))
            print("  step " + str(t) + "/" + str(STEPS) + " | CLIP similarity: " + str(round(sim, 4)))

    results.append((prompt, x.detach().cpu(), sims))
    gain = sims[-1][1] - sims[0][1]
   

In [ ]:

model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai")
model = model.to(device).eval()
tokenizer = open_clip.get_tokenizer("ViT-B-32")
for p in model.parameters():
    p.requires_grad = False
total = sum(p.numel() for p in model.parameters())
print(f"CLIP loaded: {total:,} params, 0 trained")

# Load 100 real X-rays for kernel adaptation
IMG_SIZE = 64
DATASET = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train/NORMAL"

if not os.path.exists(DATASET):
    raise FileNotFoundError("Add dataset first! Sidebar -> + Add Data -> chest-xray-pneumonia")

paths = glob.glob(f"{DATASET}/*.jpeg") + glob.glob(f"{DATASET}/*.jpg")
random.seed(42)
paths = random.sample(paths, min(100, len(paths)))

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

imgs = []
for p in paths:
    try:
        imgs.append(tfm(Image.open(p).convert("RGB")))
    except:
        pass
X_med = torch.stack(imgs).to(device)
print(f"Loaded {len(X_med)} real X-rays (kernel only, NOT training)")


CLIP_MEAN = torch.tensor([0.48145466, 0.4578275,  0.40821073]).view(1,3,1,1).to(device)
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1,3,1,1).to(device)

def clip_feats(imgs_b):
    r = torch.nn.functional.interpolate(imgs_b, (224,224), mode="bilinear", align_corners=False)
    n = (r.clamp(0,1) - CLIP_MEAN) / CLIP_STD
    f = model.encode_image(n)
    return f / f.norm(dim=-1, keepdim=True)

with torch.no_grad():
    feats = torch.cat([clip_feats(X_med[i:i+20]) for i in range(0, len(X_med), 20)])
    diff  = feats.unsqueeze(0) - feats.unsqueeze(1)
    K     = torch.exp(-(diff**2).sum(-1) / (2*0.5**2))
    alpha = torch.linalg.solve(
        K + 1e-3 * torch.eye(len(X_med), device=device),
        torch.ones(len(X_med), 1, device=device)
    )


In [ ]:

gains  = [r[2][-1][1]-r[2][0][1] for r in results]
finals = [r[2][-1][1] for r in results]

caption = f"""
Healthy lungs  -> CLIP score: {finals[0]:.3f} (+{gains[0]:.4f})
Pneumonia      -> CLIP score: {finals[1]:.3f} (+{gains[1]:.4f})
Lung nodule    -> CLIP score: {finals[2]:.3f} (+{gains[2]:.4f})
"""

print(caption)

In [ ]:


BG, ACCENT, DIM = "#0d0d0d", "#00ff88", "#888888"
LABELS = ["Healthy Lungs", "Pneumonia", "Lung Nodule"]

fig = plt.figure(figsize=(20, 14), facecolor=BG)
fig.suptitle(
    "SCORENET-ZERO  |  3 Medical Images  |  ZERO Training  |  Frozen CLIP + Steins Identity",
    color="white", fontsize=15, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.3,
                       top=0.92, bottom=0.08, left=0.05, right=0.97)

for i, (prompt, img_t, sims) in enumerate(results):

    # Generated image
    ax_img = fig.add_subplot(gs[0, i])
    ax_img.set_facecolor(BG)
    img_np = img_t.squeeze(0).permute(1,2,0).numpy()
    gray   = 0.299*img_np[:,:,0] + 0.587*img_np[:,:,1] + 0.114*img_np[:,:,2]
    ax_img.imshow(gray, cmap="bone", vmin=0, vmax=1)
    ax_img.set_title("Image " + str(i+1) + ": " + LABELS[i],
                     color=ACCENT, fontsize=13, fontweight="bold", pad=10)

    gain = sims[-1][1] - sims[0][1]
    badge_text = "CLIP: " + str(round(sims[-1][1], 3)) + "\n+" + str(round(gain, 4))
    ax_img.text(0.04, 0.05, badge_text,
        transform=ax_img.transAxes, color=ACCENT, fontsize=9,
        fontweight="bold", fontfamily="monospace",
        bbox=dict(facecolor="black", edgecolor=ACCENT, boxstyle="round,pad=0.3"))
    ax_img.axis("off")
    for sp in ax_img.spines.values():
        sp.set_edgecolor(ACCENT)
        sp.set_linewidth(2)

    ax_c = fig.add_subplot(gs[1, i])
    ax_c.set_facecolor("#111111")
    for sp in ax_c.spines.values():
        sp.set_edgecolor("#333333")

    xs = [s for s,_ in sims]
    ys = [v for _,v in sims]
    ax_c.fill_between(xs, min(ys), ys, alpha=0.15, color=ACCENT)
    ax_c.plot(xs, ys, color=ACCENT, linewidth=2.5)
    ax_c.scatter(xs[0],  ys[0],  color="white", s=70, zorder=5)
    ax_c.scatter(xs[-1], ys[-1], color=ACCENT,  s=90, zorder=5)
    ax_c.annotate(str(round(ys[0], 3)),
        xy=(xs[0], ys[0]), xytext=(15,-18), textcoords="offset points",
        color="white", fontsize=8,
        arrowprops=dict(arrowstyle="->", color="white", lw=1))
    ax_c.annotate(str(round(ys[-1], 3)),
        xy=(xs[-1], ys[-1]), xytext=(-30,12), textcoords="offset points",
        color=ACCENT, fontsize=8, fontweight="bold",
        arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1))
    ax_c.set_title("Quality Curve - Image " + str(i+1),
                   color="white", fontsize=10, fontweight="bold", pad=8)
    ax_c.set_xlabel("Step", color=DIM, fontsize=9)
    ax_c.set_ylabel("CLIP Similarity", color=DIM, fontsize=9)
    ax_c.tick_params(colors=DIM, labelsize=8)
    ax_c.set_xlim(0, STEPS)
    ax_c.grid(True, alpha=0.1, color="white")

gains  = [r[2][-1][1]-r[2][0][1] for r in results]
finals = [r[2][-1][1] for r in results]

footer = ("Dataset: chest-xray-pneumonia (100 images, kernel only, NO training)  |  "
          "CLIP gains: +" + str(round(gains[0],4)) + " | +" + str(round(gains[1],4)) + " | +" + str(round(gains[2],4))
          + "  |  Params trained: 0  |  Steps: " + str(STEPS))

fig.text(0.5, 0.01, footer, color=ACCENT, fontsize=8.5, ha="center",
         fontfamily="monospace",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#111111",
                   edgecolor=ACCENT, linewidth=1.2))

plt.savefig("scorenet_zero_3images.png", dpi=160,
            bbox_inches="tight", facecolor=BG, edgecolor="none")
plt.show()
